# 01 — Disentangled IVE Model Demo

This notebook demonstrates the active inference model of the Identified Victim Effect (IVE) with three separable mechanisms:

| Parameter | Interpretation | Neural proxy |
|-----------|---------------|-------------|
| `delta_C` | Preference shift: identified victims valued more | Insula (affective salience) |
| `delta_gamma` | Precision shift: more urgent policy selection | Striatum / dACC (gain) |
| `delta_p` | Controllability shift: higher perceived efficacy | Agency / mPFC |

We show:
1. How each mechanism independently affects P(Help)
2. Parameter sweeps for each mechanism
3. Interaction effects between mechanisms

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 12, 'figure.dpi': 120})

from ive.agent import build_agent, choose_action, get_help_probability, DEFAULTS
from ive.plotting import plot_help_rates_bar, plot_help_vs_parameter, plot_ive_delta

## 1. Single-trial demo

Build agents for statistical and identified contexts, inspect the expected free energy and action probabilities.

In [ ]:
for ctx in ['stat', 'id']:
    agent = build_agent(context=ctx)
    agent.reset()
    ctx_obs = 1 if ctx == 'id' else 0
    agent.infer_states([ctx_obs, 0, 0])
    agent.infer_policies()
    
    print(f'Context: {ctx}')
    print(f'  G (expected free energy): NoHelp={agent.G[0]:.3f}, Help={agent.G[1]:.3f}')
    print(f'  q_pi: P(NoHelp)={agent.q_pi[0]:.3f}, P(Help)={agent.q_pi[1]:.3f}')
    print(f'  Preferences C[outcome]: {agent.C[1]}')
    print()

## 2. Monte Carlo: default parameters

Run 1000 trials per context with default parameters.

In [ ]:
np.random.seed(42)
n = 1000

p_stat = get_help_probability(context='stat', n_samples=n)
p_id = get_help_probability(context='id', n_samples=n)

print(f'P(Help | statistical) = {p_stat:.3f}')
print(f'P(Help | identified)  = {p_id:.3f}')
print(f'IVE delta             = {p_id - p_stat:+.3f}')

fig, ax = plt.subplots(figsize=(4, 4))
plot_help_rates_bar(p_stat, p_id, title='Default parameters')
plt.tight_layout()
plt.show()

## 3. Parameter sweeps: isolating each mechanism

### 3a. Sweep delta_C (preference / affective valuation shift)

In [ ]:
delta_C_values = np.linspace(0.0, 2.0, 15)
n_mc = 500

h_stat_dC = []
h_id_dC = []

for dC in delta_C_values:
    ps = get_help_probability(delta_C=dC, delta_p=0.0, delta_gamma=0.0,
                              context='stat', n_samples=n_mc)
    pi = get_help_probability(delta_C=dC, delta_p=0.0, delta_gamma=0.0,
                              context='id', n_samples=n_mc)
    h_stat_dC.append(ps)
    h_id_dC.append(pi)

h_stat_dC = np.array(h_stat_dC)
h_id_dC = np.array(h_id_dC)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_help_vs_parameter(delta_C_values, h_stat_dC, h_id_dC,
                       param_name='delta_C (preference shift)', ax=axes[0])
axes[0].set_title('P(Help) vs delta_C\n(delta_p=0, delta_gamma=0)')

plot_ive_delta(delta_C_values, h_stat_dC, h_id_dC,
              param_name='delta_C', ax=axes[1])
axes[1].set_title('IVE magnitude vs delta_C')
plt.tight_layout()
plt.show()

### 3b. Sweep delta_p (controllability / perceived efficacy shift)

In [ ]:
delta_p_values = np.linspace(0.0, 0.6, 12)

h_stat_dp = []
h_id_dp = []

for dp in delta_p_values:
    ps = get_help_probability(delta_C=0.0, delta_p=dp, delta_gamma=0.0,
                              context='stat', n_samples=n_mc)
    pi = get_help_probability(delta_C=0.0, delta_p=dp, delta_gamma=0.0,
                              context='id', n_samples=n_mc)
    h_stat_dp.append(ps)
    h_id_dp.append(pi)

h_stat_dp = np.array(h_stat_dp)
h_id_dp = np.array(h_id_dp)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_help_vs_parameter(delta_p_values, h_stat_dp, h_id_dp,
                       param_name='delta_p (controllability shift)', ax=axes[0])
axes[0].set_title('P(Help) vs delta_p\n(delta_C=0, delta_gamma=0)')

plot_ive_delta(delta_p_values, h_stat_dp, h_id_dp,
              param_name='delta_p', ax=axes[1])
axes[1].set_title('IVE magnitude vs delta_p')
plt.tight_layout()
plt.show()

### 3c. Sweep delta_gamma (precision / urgency shift)

In [ ]:
delta_gamma_values = np.linspace(0.0, 4.0, 12)

h_stat_dg = []
h_id_dg = []

for dg in delta_gamma_values:
    ps = get_help_probability(delta_C=0.0, delta_p=0.0, delta_gamma=dg,
                              context='stat', n_samples=n_mc)
    pi = get_help_probability(delta_C=0.0, delta_p=0.0, delta_gamma=dg,
                              context='id', n_samples=n_mc)
    h_stat_dg.append(ps)
    h_id_dg.append(pi)

h_stat_dg = np.array(h_stat_dg)
h_id_dg = np.array(h_id_dg)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_help_vs_parameter(delta_gamma_values, h_stat_dg, h_id_dg,
                       param_name='delta_gamma (precision shift)', ax=axes[0])
axes[0].set_title('P(Help) vs delta_gamma\n(delta_C=0, delta_p=0)')

plot_ive_delta(delta_gamma_values, h_stat_dg, h_id_dg,
              param_name='delta_gamma', ax=axes[1])
axes[1].set_title('IVE magnitude vs delta_gamma')
plt.tight_layout()
plt.show()

## 4. Interaction: delta_C x cost_penalty

The IVE emerges most strongly when cost is in a critical range where statistical victims are marginal but identified victims cross the help threshold.

In [ ]:
cost_values = np.linspace(0.5, 3.0, 12)
delta_C_vals = [0.0, 0.5, 1.0, 1.5]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for dC in delta_C_vals:
    ive_curve = []
    for cost in cost_values:
        ps = get_help_probability(delta_C=dC, delta_p=0.0, delta_gamma=0.0,
                                  cost_penalty=cost, context='stat', n_samples=400)
        pi = get_help_probability(delta_C=dC, delta_p=0.0, delta_gamma=0.0,
                                  cost_penalty=cost, context='id', n_samples=400)
        ive_curve.append(pi - ps)
    axes[0].plot(cost_values, ive_curve, label=f'dC={dC}')

axes[0].set_xlabel('Cost penalty')
axes[0].set_ylabel('IVE magnitude (P_id - P_stat)')
axes[0].set_title('IVE vs cost at different delta_C')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(0, color='k', linestyle=':', alpha=0.3)

# Heatmap: delta_C x cost -> IVE magnitude
dC_grid = np.linspace(0.0, 2.0, 10)
cost_grid = np.linspace(0.5, 3.0, 10)
ive_heat = np.zeros((len(cost_grid), len(dC_grid)))

for i, cost in enumerate(cost_grid):
    for j, dC in enumerate(dC_grid):
        ps = get_help_probability(delta_C=dC, delta_p=0.0, cost_penalty=cost,
                                  context='stat', n_samples=300)
        pi = get_help_probability(delta_C=dC, delta_p=0.0, cost_penalty=cost,
                                  context='id', n_samples=300)
        ive_heat[i, j] = pi - ps

im = axes[1].imshow(ive_heat, origin='lower', aspect='auto',
                     extent=[dC_grid[0], dC_grid[-1], cost_grid[0], cost_grid[-1]],
                     cmap='RdBu_r', vmin=-0.3, vmax=0.3)
plt.colorbar(im, ax=axes[1], label='IVE (P_id - P_stat)')
axes[1].set_xlabel('delta_C')
axes[1].set_ylabel('Cost penalty')
axes[1].set_title('IVE heatmap: delta_C x cost')

plt.tight_layout()
plt.show()

## 5. Key takeaways

1. **delta_C** (preference shift) is the primary driver of behavioural IVE — it shifts the expected utility of saving the identified victim
2. **delta_p** (controllability) amplifies the effect by making help seem more effective for identified victims
3. **delta_gamma** (precision) has a nuanced effect — it makes the agent more deterministic toward whichever action is already preferred, which can either amplify or attenuate the IVE depending on the regime
4. The IVE is strongest at **intermediate cost** — when helping is marginal for statistical victims but still worthwhile for identified ones